# BT4012 Competition — Data Preprocessing

Continues from `01_eda.ipynb`. Builds the feature tables used by every later notebook:

1. **Graph features** from the transaction edge list: degree, centrality, community and neighborhood features. None of them use labels, so test rows get them too.
2. **Feature reduction**: drop near-duplicate features (step 1), then prune the remaining multicollinearity with iterative VIF (step 2). Saves `data/processed/train_reduced.csv` and `data/processed/test_reduced.csv`.

3. **Label propagation** on the unlabeled rows: adds `label_propagated` / `label_propagated_confidence` next to the original `label`. `03_supervised_learning.ipynb` (step 2) tests whether adding these pseudo-labels to the training data helps.

In [1]:
import warnings

import networkx as nx
import numpy as np
import pandas as pd
from scipy.sparse.csgraph import shortest_path
from sklearn.exceptions import ConvergenceWarning
from sklearn.semi_supervised import LabelPropagation

pd.set_option("display.max_columns", 20)

## Load data

In [2]:
train = pd.read_csv("data/raw/train.csv")
test = pd.read_csv("data/raw/test.csv")

feat_cols = [c for c in train.columns if c.startswith("feat_")]

print("train:", train.shape)
print("test:", test.shape)
print("num feature columns:", len(feat_cols))

train: (141772, 168)
test: (15329, 168)
num feature columns: 165


## New feature — transaction in/out-degree

Added because the transaction graph shows how many transactions sent money *into* a given transaction (in-degree) versus how many it sent money *out to* (out-degree) — a transaction that mostly receives (consolidation, e.g. a mixing service) looks different from one that mostly distributes (fan-out/structuring), which is a natural fraud signal the 165 anonymized features don't directly expose.

We use `total_degree` (in+out) and `degree_diff` (out-in) rather than `in_degree`/`out_degree`/`degree_diff` together, since the third is an exact linear function of the other two and breaks VIF. Added to `feat_cols` before Step 1, so both go through the same dedup/VIF pipeline as the other features.

In [3]:
edges = pd.read_csv("data/raw/txs_edgelist.csv")

in_degree = edges["txId2"].value_counts()
out_degree = edges["txId1"].value_counts()

def add_degree_features(df):
    df = df.copy()
    in_deg = df["txId"].map(in_degree).fillna(0).astype(int)
    out_deg = df["txId"].map(out_degree).fillna(0).astype(int)
    df["total_degree"] = in_deg + out_deg
    df["degree_diff"] = out_deg - in_deg
    return df

train = add_degree_features(train)
test = add_degree_features(test)

degree_cols = ["total_degree", "degree_diff"]
feat_cols = feat_cols + degree_cols

print(train[degree_cols].describe())
print()
print("correlation between total_degree and degree_diff:", train["total_degree"].corr(train["degree_diff"]))
print()
labeled_for_check = train[train["label"].notna()]
print("correlation with label:")
print(labeled_for_check[degree_cols].corrwith(labeled_for_check["label"]).abs())

        total_degree    degree_diff
count  141772.000000  141772.000000
mean        2.302204       0.000000
std         4.045061       4.097831
min         1.000000    -284.000000
25%         1.000000       0.000000
50%         2.000000       0.000000
75%         2.000000       1.000000
max       473.000000     471.000000

correlation between total_degree and degree_diff: -0.5361395151386112

correlation with label:
total_degree    0.043815
degree_diff     0.008021
dtype: float64


## New features — graph centrality

Degree (above) only counts a transaction's direct connections. Centrality measures its *position* in the transaction network, computed over the whole graph:

| Feature | Question it answers |
|---|---|
| `betweenness` | How often does this transaction sit on the shortest path between two others? High for bridges between groups, e.g. a layering step between a source and a mixer. |
| `closeness` | How few hops does it take to reach every other transaction it's connected to? |
| `pagerank` | Is it connected to other well-connected transactions? |

- **Betweenness** is estimated from 5,000 sampled source nodes. The exact version is too slow in pure `networkx` at ~200k nodes.
- **PageRank replaces eigenvector centrality.** The graph is 49 disconnected pieces, one per `time_step`, and eigenvector centrality puts almost all its weight on the largest piece, giving ~0 to every transaction in the other 48. PageRank answers the same question and is well-defined on a graph in pieces. It uses the edge direction (money flow); betweenness and closeness treat edges as undirected.
- **Closeness** uses `networkx`'s whole-graph formula (`wf_improved`). Paths never cross between pieces, so the BFS runs piece by piece purely for speed; the values are identical to computing it on the whole graph at once.

All three are added to `feat_cols` before Step 1, so they go through the same dedup/VIF pipeline as every other feature. Nothing here uses labels, so the same values are available for test rows.

In [4]:
G_undirected = nx.from_pandas_edgelist(edges, "txId1", "txId2")
G_directed = nx.from_pandas_edgelist(edges, "txId1", "txId2", create_using=nx.DiGraph)
N = G_undirected.number_of_nodes()

betweenness = nx.betweenness_centrality(G_undirected, k=5000, seed=42)

closeness = {}
for piece in nx.connected_components(G_undirected):
    nodes = list(piece)
    n = len(nodes)
    A = nx.to_scipy_sparse_array(G_undirected.subgraph(nodes), nodelist=nodes, format="csr")
    for start in range(0, n, 1000):
        idx = np.arange(start, min(start + 1000, n))
        dist_sum = shortest_path(A, unweighted=True, directed=False, indices=idx).sum(axis=1)
        for i, s in zip(idx, dist_sum):
            # networkx wf_improved closeness: (n-1)/sum(dist), scaled by the share of the graph reachable
            closeness[nodes[i]] = ((n - 1) / s) * ((n - 1) / (N - 1)) if s > 0 else 0.0

pagerank = nx.pagerank(G_directed)


def add_centrality_features(df):
    df = df.copy()
    df["betweenness"] = df["txId"].map(betweenness).fillna(0.0)
    df["closeness"] = df["txId"].map(closeness).fillna(0.0)
    df["pagerank"] = df["txId"].map(pagerank).fillna(0.0)
    return df


train = add_centrality_features(train)
test = add_centrality_features(test)

centrality_cols = ["betweenness", "closeness", "pagerank"]
feat_cols = feat_cols + centrality_cols

print(train[centrality_cols].describe())
print()
labeled_for_check = train[train["label"].notna()]
print("correlation with label:")
print(labeled_for_check[centrality_cols].corrwith(labeled_for_check["label"]).abs())

        betweenness      closeness       pagerank
count  1.417720e+05  141772.000000  141772.000000
mean   1.945318e-06       0.001852       0.000005
std    1.041042e-05       0.001087       0.000009
min    0.000000e+00       0.000046       0.000002
25%    0.000000e+00       0.001029       0.000002
50%    1.531168e-07       0.001654       0.000003
75%    7.695070e-07       0.002432       0.000006
max    6.146254e-04       0.006284       0.000527

correlation with label:
betweenness    0.045739
closeness      0.153522
pagerank       0.033853
dtype: float64


## New features — community ("island") structure

**Louvain** community detection splits the transaction network into communities: groups with many links among themselves and few to the rest. It maximizes *modularity*, the share of edges inside groups minus what random wiring would give, by repeatedly moving each transaction into whichever neighboring group raises modularity most, then merging groups into super-nodes and repeating. It runs once on the whole graph, undirected, with the default `resolution=1.0` and a fixed `seed` so the groups are reproducible. It uses no labels.

Louvain's output is only a group ID per transaction, which is arbitrary and not used as a feature. Instead, each transaction gets three numbers describing its group:

| Feature | Formula | Meaning |
|---|---|---|
| `community_size` | transactions in the group | Small, tight group vs. large one |
| `community_density` | edges inside the group ÷ transactions in the group | ≈1 for a chain or tree; well above 1 means clique-like cross-links |
| `community_internal_ratio` | edges inside ÷ (edges inside + edges leaving) | 1.0 = a closed group that only transacts with itself |

(Island metrics at the level of *connected components* aren't usable here: each component is exactly one `time_step`, so they'd just encode the time period.)

In [5]:
communities = nx.community.louvain_communities(G_undirected, resolution=1.0, seed=42)

community_size, community_density, community_internal_ratio = {}, {}, {}
for members in communities:
    size = len(members)
    internal = G_undirected.subgraph(members).number_of_edges()
    leaving = sum(1 for u in members for v in G_undirected.neighbors(u) if v not in members)
    density = internal / size
    internal_ratio = internal / (internal + leaving) if internal + leaving else 0.0
    for node in members:
        community_size[node] = size
        community_density[node] = density
        community_internal_ratio[node] = internal_ratio

sizes = pd.Series([len(c) for c in communities])
print(f"{len(communities)} communities; size median {sizes.median():.0f}, min {sizes.min()}, max {sizes.max()}")


def add_community_features(df):
    df = df.copy()
    df["community_size"] = df["txId"].map(community_size)
    df["community_density"] = df["txId"].map(community_density)
    df["community_internal_ratio"] = df["txId"].map(community_internal_ratio)
    return df


train = add_community_features(train)
test = add_community_features(test)

community_cols = ["community_size", "community_density", "community_internal_ratio"]
assert train[community_cols].notna().all().all() and test[community_cols].notna().all().all()
feat_cols = feat_cols + community_cols

print()
print(train[community_cols].describe())
print()
labeled_for_check = train[train["label"].notna()]
print("correlation with label:")
print(labeled_for_check[community_cols].corrwith(labeled_for_check["label"]).abs())

308 communities; size median 73, min 17, max 6735

       community_size  community_density  community_internal_ratio
count   141772.000000      141772.000000             141772.000000
mean      3042.319696           1.146291                  0.991635
std       1700.912228           0.281664                  0.009246
min         20.000000           0.950000                  0.942857
25%       1976.000000           1.070850                  0.988132
50%       3197.000000           1.096318                  0.995619
75%       4227.000000           1.147533                  0.997368
max       6385.000000           5.155963                  1.000000

correlation with label:
community_size              0.146730
community_density           0.079906
community_internal_ratio    0.018704
dtype: float64


## New features — neighborhood

Three features describing a transaction's immediate surroundings, beyond its own degree:

| Feature | Formula | Meaning |
|---|---|---|
| `flow_ratio` | `degree_diff ÷ total_degree` | −1 = only receives (consolidation), +1 = only sends (fan-out), 0 = balanced. Unlike `degree_diff`, it doesn't depend on how many connections the transaction has. |
| `mean_neighbor_degree` | average `total_degree` of its direct neighbors | Is it surrounded by busy transactions? |
| `hop2_reach` | transactions within 2 hops | How large its local neighborhood is |

(The highest neighbor degree was also tried, but it is almost a duplicate of `hop2_reach` (|r| ≥ 0.9) and the duplicate filter below always dropped it, so it isn't computed.) Nothing here uses labels, so test rows get the same features.

In [6]:
total_degree_lookup = in_degree.add(out_degree, fill_value=0).to_dict()

mean_neighbor_degree, hop2_reach = {}, {}
for node in G_undirected.nodes:
    one_hop = set(G_undirected.neighbors(node))
    neighbor_degrees = [total_degree_lookup[n] for n in one_hop]
    mean_neighbor_degree[node] = sum(neighbor_degrees) / len(neighbor_degrees)
    two_hop = set().union(*(G_undirected.neighbors(n) for n in one_hop)) - one_hop - {node}
    hop2_reach[node] = len(one_hop) + len(two_hop)


def add_neighborhood_features(df):
    df = df.copy()
    df["flow_ratio"] = df["degree_diff"] / df["total_degree"]
    df["mean_neighbor_degree"] = df["txId"].map(mean_neighbor_degree)
    df["hop2_reach"] = df["txId"].map(hop2_reach)
    return df


train = add_neighborhood_features(train)
test = add_neighborhood_features(test)

neighborhood_cols = ["flow_ratio", "mean_neighbor_degree", "hop2_reach"]
assert train[neighborhood_cols].notna().all().all() and test[neighborhood_cols].notna().all().all()
feat_cols = feat_cols + neighborhood_cols

print(train[neighborhood_cols].describe())
print()
labeled_for_check = train[train["label"].notna()]
print("correlation with label:")
print(labeled_for_check[neighborhood_cols].corrwith(labeled_for_check["label"]).abs())

          flow_ratio  mean_neighbor_degree     hop2_reach
count  141772.000000         141772.000000  141772.000000
mean        0.100058             11.265520      20.748794
std         0.672927             28.745028      48.753688
min        -1.000000              1.058824       2.000000
25%         0.000000              2.000000       4.000000
50%         0.000000              2.666667       5.000000
75%         1.000000              6.500000      14.000000
max         1.000000            473.000000     961.000000

correlation with label:
flow_ratio              0.033036
mean_neighbor_degree    0.023413
hop2_reach              0.056598
dtype: float64


## Label relevance (used to break ties below)

When two features are near-duplicates, or several are jointly collinear, we don't want to drop the one that happens to come first alphabetically/numerically — we want to keep whichever is more useful for predicting `label`. Compute `|corr|` with `label` once, on the labeled subset (22% of train), and reuse it as the tie-break in both Step 1 and Step 2 below.

In [7]:
labeled = train.dropna(subset=["label"])
label_corr = labeled[feat_cols].corrwith(labeled["label"]).abs()

label_corr.sort_values(ascending=False).head(10)

feat_53     0.307551
feat_89     0.266436
feat_55     0.261658
feat_90     0.256831
feat_91     0.223915
feat_148    0.196755
feat_150    0.186980
feat_52     0.186078
feat_142    0.180250
feat_54     0.179958
dtype: float64

## Step 1 — Drop near-duplicate features

The EDA found 147 feature pairs with `|r| >= 0.9` (up from 103 at the stricter `|r| > 0.95` cutoff). Feeding these into VIF as-is produces singular/near-singular matrices and blows up to infinity.

(A stricter 0.8 threshold was also tried; it didn't improve validation scores, so 0.9 is used.)

Naively keeping "whichever column comes first" is arbitrary and can throw away the more predictive feature of a pair. Instead:

1. Group features into connected components using an `|r| >= 0.9` adjacency graph — this also correctly handles chains of 3+ mutually-duplicate features, not just isolated pairs.
2. Within each group, keep the single feature with the **highest `|corr|` with `label`** (the linear correlation-with-label priority from `01_eda.ipynb`, computed above on train only — test never informs this decision) and drop the rest of the group.

In [8]:
DUPLICATE_THRESHOLD = 0.9

feat_corr = train[feat_cols].corr()
abs_corr_matrix = feat_corr.abs()
upper = abs_corr_matrix.where(np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool))

dupe_graph = nx.Graph()
dupe_graph.add_nodes_from(feat_cols)
pairs_above_threshold = upper.stack()
pairs_above_threshold = pairs_above_threshold[pairs_above_threshold >= DUPLICATE_THRESHOLD]
dupe_graph.add_edges_from(pairs_above_threshold.index)

duplicate_groups = [g for g in nx.connected_components(dupe_graph) if len(g) > 1]

to_drop_duplicates = []
for group in duplicate_groups:
    ranked = label_corr.loc[list(group)].sort_values(ascending=False)
    keeper = ranked.index[0]
    to_drop_duplicates.extend([c for c in group if c != keeper])

print(f"duplicate groups found: {len(duplicate_groups)}")
print(f"near-duplicate columns to drop (|r| >= {DUPLICATE_THRESHOLD}): {len(to_drop_duplicates)}")
to_drop_duplicates

duplicate groups found: 42
near-duplicate columns to drop (|r| >= 0.9): 78


['feat_1',
 'feat_8',
 'feat_19',
 'feat_11',
 'feat_9',
 'feat_17',
 'feat_6',
 'feat_14',
 'feat_13',
 'feat_21',
 'feat_28',
 'feat_29',
 'feat_31',
 'feat_23',
 'feat_30',
 'feat_26',
 'feat_33',
 'feat_27',
 'feat_37',
 'feat_34',
 'feat_39',
 'feat_77',
 'feat_78',
 'feat_47',
 'feat_84',
 'feat_40',
 'feat_79',
 'feat_46',
 'feat_49',
 'feat_43',
 'feat_41',
 'feat_85',
 'feat_51',
 'feat_44',
 'feat_50',
 'feat_55',
 'feat_52',
 'feat_57',
 'feat_58',
 'feat_61',
 'feat_60',
 'feat_66',
 'feat_65',
 'feat_67',
 'feat_69',
 'feat_62',
 'feat_68',
 'feat_72',
 'feat_73',
 'feat_74',
 'feat_81',
 'feat_87',
 'feat_86',
 'feat_91',
 'feat_92',
 'feat_97',
 'feat_94',
 'feat_99',
 'feat_101',
 'feat_100',
 'feat_104',
 'feat_111',
 'feat_115',
 'feat_113',
 'feat_116',
 'feat_118',
 'feat_122',
 'feat_128',
 'feat_132',
 'feat_133',
 'feat_135',
 'feat_139',
 'feat_140',
 'feat_146',
 'feat_152',
 'feat_155',
 'feat_159',
 'feat_165']

In [9]:
feat_cols_deduped = [c for c in feat_cols if c not in to_drop_duplicates]
print(f"features remaining after dedup: {len(feat_cols_deduped)} / {len(feat_cols)}")

features remaining after dedup: 98 / 176


## Step 2 — Iterative VIF pruning

Pairwise dedup only catches two-column collinearity. VIF regresses each feature on all others to catch multi-feature collinearity (e.g. an aggregate that's a linear combination of several local features).

Same concern as Step 1 applies here: a feature can be both highly collinear *and* highly predictive, so each round we drop the **least label-relevant** feature among those above the VIF threshold — not necessarily the highest-VIF one. Repeat until none remain above threshold.

**Implementation note:** fitting `statsmodels`' `variance_inflation_factor` per column is far too slow here (~92s per pass, ~17-20 min total). Instead we use `VIF_i = diag(inv(R))[i]`, where `R` is the correlation matrix — mathematically identical (verified to 9 decimal places) but ~0.03s total. This only works because all features are numerical/continuous; it wouldn't apply as-is to categorical columns.

A VIF of 5–10 is the usual cutoff; using 10 here since the aggregated features are expected to correlate with local features by construction.

In [10]:
VIF_THRESHOLD = 10.0

def compute_vif(corr_df, cols):
    sub_corr = corr_df.loc[cols, cols].to_numpy()
    return pd.Series(np.diag(np.linalg.inv(sub_corr)), index=cols)

vif_cols = list(feat_cols_deduped)
dropped_for_vif = []

while True:
    vif = compute_vif(feat_corr, vif_cols)
    above_threshold = vif[vif > VIF_THRESHOLD]
    if above_threshold.empty:
        break
    # Among the redundant (above-threshold) features, drop the one least useful
    # for the label — not simply the one with the highest VIF.
    drop_col = label_corr.loc[above_threshold.index].idxmin()
    vif_cols.remove(drop_col)
    dropped_for_vif.append((drop_col, vif[drop_col], label_corr[drop_col]))
    print(
        f"dropping {drop_col} (VIF={vif[drop_col]:.1f}, |label corr|={label_corr[drop_col]:.3f}), "
        f"{len(vif_cols)} features remaining"
    )

print(f"\nfinal feature count after VIF pruning: {len(vif_cols)} / {len(feat_cols_deduped)}")

dropping feat_145 (VIF=82.0, |label corr|=0.016), 97 features remaining
dropping feat_149 (VIF=73.7, |label corr|=0.026), 96 features remaining
dropping feat_162 (VIF=24.3, |label corr|=0.026), 95 features remaining
dropping feat_157 (VIF=23.6, |label corr|=0.035), 94 features remaining
dropping feat_106 (VIF=24.9, |label corr|=0.043), 93 features remaining
dropping feat_124 (VIF=16.0, |label corr|=0.055), 92 features remaining
dropping feat_143 (VIF=12.7, |label corr|=0.059), 91 features remaining
dropping feat_137 (VIF=34.4, |label corr|=0.067), 90 features remaining
dropping feat_24 (VIF=96.1, |label corr|=0.073), 89 features remaining
dropping feat_127 (VIF=10.9, |label corr|=0.083), 88 features remaining
dropping feat_107 (VIF=11.4, |label corr|=0.089), 87 features remaining
dropping feat_151 (VIF=31.6, |label corr|=0.090), 86 features remaining
dropping feat_90 (VIF=13.5, |label corr|=0.257), 85 features remaining

final feature count after VIF pruning: 85 / 98


In [11]:
final_vif = compute_vif(feat_corr, vif_cols).sort_values(ascending=False)
print("max remaining VIF:", final_vif.max())
final_vif.head(10)

max remaining VIF: 7.638828843638012


feat_48                 7.638829
feat_25                 6.860502
feat_22                 6.327524
feat_119                4.775419
feat_103                4.722886
hop2_reach              4.705586
feat_42                 4.292990
mean_neighbor_degree    4.158950
feat_136                4.098189
total_degree            3.661589
dtype: float64

## Save reduced feature set

In [12]:
keep_cols = ["txId", "time_step"] + vif_cols

train_reduced = train[keep_cols + ["label"]].copy()
test_reduced = test[keep_cols].copy()

train_reduced.to_csv("data/processed/train_reduced.csv", index=False)
test_reduced.to_csv("data/processed/test_reduced.csv", index=False)

print("saved data/processed/train_reduced.csv:", train_reduced.shape)
print("saved data/processed/test_reduced.csv:", test_reduced.shape)

saved data/processed/train_reduced.csv: (141772, 88)
saved data/processed/test_reduced.csv: (15329, 87)


## Label propagation for the unlabeled rows

78% of `train` rows have `label = NaN`. **Label propagation** (`sklearn.semi_supervised.LabelPropagation`, `kernel="knn"`, 7 neighbours, over the 85 reduced features) spreads the known labels to unlabeled transactions with similar features, giving every unlabeled row a predicted label and a confidence. All 31,235 known labels are used as anchors.

The predictions are saved as two new columns, `label_propagated` and `label_propagated_confidence`, rather than overwriting `label`, so genuine labels stay distinguishable from pseudo-labels. `03_supervised_learning.ipynb` (step 2) tests whether adding confident pseudo-labels to the training data helps.

In [13]:
# LabelPropagation's tolerance-based convergence check often doesn't trigger within max_iter here;
# the warning is expected.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

labeled_mask = train_reduced["label"].notna()
labeled_idx = train_reduced.index[labeled_mask]
unlabeled_idx = train_reduced.index[~labeled_mask]

print("labeled rows:", len(labeled_idx))
print("unlabeled rows:", len(unlabeled_idx))

labeled rows: 31235
unlabeled rows: 110537


## Apply to the real unlabeled rows

Fit on all 31,235 known labels plus the genuinely unlabeled rows, and report how many unlabeled rows reach a confidence of at least 0.9. Saved as two new columns — `label_propagated` and `label_propagated_confidence` — rather than overwriting `label` itself, so the genuine ground truth stays distinguishable from a confident guess.

In [14]:
PROPAGATION_THRESHOLD = 0.9

X_full_prop = train_reduced[vif_cols].to_numpy()
y_full_prop_input = train_reduced["label"].fillna(-1).astype(int).to_numpy()

final_prop_model = LabelPropagation(kernel="knn", n_neighbors=7, max_iter=1000, n_jobs=-1)
final_prop_model.fit(X_full_prop, y_full_prop_input)

all_confidence = final_prop_model.label_distributions_.max(axis=1)
all_predicted = final_prop_model.transduction_

train_reduced["label_propagated"] = all_predicted
train_reduced["label_propagated_confidence"] = all_confidence

is_unlabeled_row = train_reduced["label"].isna()
is_confident = train_reduced["label_propagated_confidence"] >= PROPAGATION_THRESHOLD
new_confident_labels = is_unlabeled_row & is_confident

print(f"unlabeled rows: {is_unlabeled_row.sum()}")
print(f"of those, confidently labeled (>= {PROPAGATION_THRESHOLD}): {new_confident_labels.sum()} "
      f"({new_confident_labels.sum() / is_unlabeled_row.sum():.1%} coverage)")
print()
print("breakdown of the new confident pseudo-labels:")
print(train_reduced.loc[new_confident_labels, "label_propagated"].value_counts().rename({0: "licit", 1: "illicit"}))

train_reduced.to_csv("data/processed/train_reduced.csv", index=False)
print("\nsaved data/processed/train_reduced.csv with label_propagated / label_propagated_confidence columns:", train_reduced.shape)

unlabeled rows: 110537
of those, confidently labeled (>= 0.9): 48014 (43.4% coverage)

breakdown of the new confident pseudo-labels:
label_propagated
licit      41479
illicit     6535
Name: count, dtype: int64

saved data/processed/train_reduced.csv with label_propagated / label_propagated_confidence columns: (141772, 90)
